# Smart City V2 — 01 Calibration

Đo **12 cạnh + LEFT 90 + RIGHT 90** bằng đúng đường phát lệnh mà runner sử dụng. `THROTTLE = 0.50` bị khóa trong source, không có slider chỉnh ga.

> An toàn: kê bánh xe khi kiểm tra lần đầu, chỉ ARM ngay trước phép đo, luôn sẵn tay bấm **STOP & RECORD**. Mỗi phép đo nên lặp lại ít nhất 3 lần; notebook lấy median.

In [ ]:
from pathlib import Path
import copy, statistics, time
import ipywidgets as widgets
from IPython.display import display

cwd = Path.cwd().resolve()
WORKDIR = cwd if (cwd / 'smart_city_v2_core.py').exists() else cwd / 'smart city v2'
if not (WORKDIR / 'smart_city_v2_core.py').exists():
    raise FileNotFoundError('Open this notebook from yolo_lane_following or smart city v2')
import sys
if str(WORKDIR) not in sys.path: sys.path.insert(0, str(WORKDIR))
from smart_city_v2_core import FIXED_THROTTLE, EDGES, MotionDriver, default_config, load_config, save_config, edge_key, are_neighbors

CONFIG_PATH = WORKDIR / 'track_config.json'
config = load_config(CONFIG_PATH, require_measurements=False) if CONFIG_PATH.exists() else default_config()
print(f'Fixed throttle: {FIXED_THROTTLE:.2f} (LOCKED)')
print('Config:', CONFIG_PATH)

## Kết nối xe — mặc định DISARMED
Cả đo cạnh và đo quẹo đều đặt steering trước, sau đó phát đúng ga cố định `0.50`; đồng hồ bắt đầu tại thời điểm phát ga.

In [ ]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()
car.steering_gain = float(config['motion'].get('steering_gain', -0.65))
car.steering_offset = float(config['motion'].get('steering_offset', 0.0))
car.throttle_gain = float(config['motion'].get('throttle_gain', 0.8))
car.throttle = 0.0; car.steering = 0.0
arm = widgets.Checkbox(value=False, description='ARM MOTOR')
hardware_status = widgets.HTML('<b style="color:#b00">DISARMED — motor stopped</b>')
driver = MotionDriver(car, armed=lambda: bool(arm.value))

def on_arm(change):
    if not change['new']:
        driver.stop(center=True)
        hardware_status.value = '<b style="color:#b00">DISARMED — motor stopped</b>'
    else:
        hardware_status.value = '<b style="color:#b60">ARMED — ready, car is still stopped</b>'
arm.observe(on_arm, names='value')
display(widgets.VBox([arm, hardware_status]))

## Đo thực tế
- Với cạnh: chọn hai ô `FROM NODE` và `TO NODE`; đặt xe thủ công tại FROM, quay đầu xe hướng về TO, steering center.
- Với quẹo: đặt xe tại tâm giao lộ và hướng theo cạnh đi vào; bấm STOP đúng khi xe hoàn tất 90°.
- Giữ nguyên mặt bàn, pin, lốp và tải xe như lúc chạy route.

In [ ]:
measure_mode = widgets.ToggleButtons(options=[('EDGE','edge'), ('LEFT 90','turn:left_90'), ('RIGHT 90','turn:right_90')], value='edge', description='Measure')
nodes = sorted({node for pair in EDGES for node in pair})
from_node = widgets.Dropdown(options=nodes, value=7, description='FROM NODE', layout=widgets.Layout(width='190px'))
to_node = widgets.Dropdown(options=[4,8], value=4, description='TO NODE', layout=widgets.Layout(width='190px'))
placement_label = widgets.HTML()
throttle_locked = widgets.FloatText(value=FIXED_THROTTLE, description='Throttle', disabled=True)
start_button = widgets.Button(description='▶ START', button_style='success', icon='play')
stop_button = widgets.Button(description='■ STOP & RECORD', button_style='danger', icon='stop', disabled=True)
reset_samples = widgets.Button(description='Reset selected', icon='trash')
elapsed_label = widgets.HTML('<b>Elapsed: —</b>')
sample_output = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='8px'))
samples = {f'edge:{edge_key(a,b)}': [] for a,b in EDGES}
samples.update({'turn:left_90': [], 'turn:right_90': []})
active = {'started': None, 'target': None}

def selected_target():
    if measure_mode.value != 'edge': return measure_mode.value
    a, b = int(from_node.value), int(to_node.value)
    if not are_neighbors(a, b): raise ValueError(f'Node {a} và Node {b} không có cạnh nối trực tiếp')
    return f'edge:{edge_key(a,b)}'

def selected_steering(key):
    if key.startswith('edge:'): return float(globals().get('center_w', widgets.FloatText(value=config['motion']['steering_center'])).value)
    name = key.split(':', 1)[1]; widget_name = 'left_steer_w' if name == 'left_90' else 'right_steer_w'
    widget = globals().get(widget_name)
    return float(widget.value if widget is not None else config['turns'][name]['steering'])

def selected_phase_time(turn_name, phase):
    side = 'left' if turn_name == 'left_90' else 'right'
    suffix = 'pre_w' if phase == 'pre_steer_time' else 'center_w'
    widget = globals().get(f'{side}_{suffix}')
    return float(widget.value if widget is not None else config['turns'][turn_name].get(phase, 0.15))

def redraw_samples():
    try:
        key = selected_target(); values = samples[key]
        if key.startswith('edge:'):
            placement_label.value = f'<b>PLACE CAR AT NODE {from_node.value} → FACE NODE {to_node.value}</b><br>Đặt xe thủ công, sau đó bấm START.'
        else:
            side = 'LEFT' if key.endswith('left_90') else 'RIGHT'
            placement_label.value = f'<b>PLACE CAR FOR {side} 90°</b><br>Đặt xe thủ công tại tâm giao lộ, hướng theo cạnh đi vào.'
    except Exception as exc:
        placement_label.value = f'<b style="color:#b00">{exc}</b>'; return
    with sample_output:
        sample_output.clear_output(wait=True)
        print('Samples:', ', '.join(f'{x:.4f}s' for x in values) or 'none')
        if values: print(f'Median used: {statistics.median(values):.4f}s')

def start_measure(_):
    if active['started'] is not None: return
    try: key = selected_target()
    except Exception as exc:
        elapsed_label.value = f'<b style="color:#b00">{exc}</b>'; return
    try:
        if key.startswith('turn:'):
            turn_cfg = config['turns'][key.split(':',1)[1]]
            elapsed_label.value = '<b>PHASE 1/3 — PRE-STEER, throttle=0</b>'
            driver.prepare_steering(selected_steering(key))
            time.sleep(selected_phase_time(key.split(':',1)[1], 'pre_steer_time'))
        active['started'] = driver.start(selected_steering(key))
        active['target'] = key
        start_button.disabled = True; stop_button.disabled = False
        measure_mode.disabled = True; from_node.disabled = True; to_node.disabled = True
        phase = 'PHASE 2/3 — DRIVE ARC 90' if key.startswith('turn:') else 'RUN EDGE'
        elapsed_label.value = f'<b style="color:#080">{phase} — press STOP at target</b>'
    except Exception as exc:
        driver.stop(center=True); elapsed_label.value = f'<b style="color:#b00">{exc}</b>'

def stop_measure(_):
    if active['started'] is None: return
    driver.stop(center=False)
    elapsed = time.perf_counter() - active['started']
    key = active['target']; samples[key].append(elapsed)
    if key.startswith('edge:'):
        edge_name = key.split(':',1)[1]; config['edges'][edge_name] = round(statistics.median(samples[key]), 4)
        if 'edge_widgets' in globals() and edge_name in edge_widgets: edge_widgets[edge_name].value = config['edges'][edge_name]
    else:
        turn_name = key.split(':',1)[1]; config['turns'][turn_name]['time'] = round(statistics.median(samples[key]), 4)
        time_widget = globals().get('left_time_w' if turn_name == 'left_90' else 'right_time_w')
        if time_widget is not None: time_widget.value = config['turns'][turn_name]['time']
    if key.startswith('turn:'):
        turn_cfg = config['turns'][key.split(':',1)[1]]
        elapsed_label.value = '<b>PHASE 3/3 — CENTER steering=0, throttle=0</b>'
        if arm.value:
            driver.prepare_steering(0.0)
            time.sleep(selected_phase_time(key.split(':',1)[1], 'center_settle_time'))
        else:
            driver.stop(center=True)
    else:
        driver.stop(center=True)
    active.update(started=None, target=None)
    start_button.disabled = False; stop_button.disabled = True
    measure_mode.disabled = False; from_node.disabled = measure_mode.value != 'edge'; to_node.disabled = measure_mode.value != 'edge'
    elapsed_label.value = f'<b>Elapsed: {elapsed:.4f} s — recorded</b>'
    redraw_samples()

def reset_selected(_):
    if active['started'] is not None: return
    try: samples[selected_target()].clear()
    except Exception: pass
    redraw_samples()

def update_node_choices(_=None):
    if measure_mode.value == 'edge':
        from_node.disabled = False; to_node.disabled = False
        neighbors = sorted(node for node in nodes if are_neighbors(int(from_node.value), node))
        old = to_node.value; to_node.options = neighbors
        to_node.value = old if old in neighbors else neighbors[0]
    else:
        from_node.disabled = True; to_node.disabled = True
    redraw_samples()

start_button.on_click(start_measure); stop_button.on_click(stop_measure); reset_samples.on_click(reset_selected)
measure_mode.observe(update_node_choices, names='value')
from_node.observe(update_node_choices, names='value'); to_node.observe(lambda _: redraw_samples(), names='value')
update_node_choices()
display(widgets.VBox([widgets.HBox([measure_mode, throttle_locked]), widgets.HBox([from_node, to_node]), placement_label, widgets.HBox([start_button, stop_button, reset_samples]), elapsed_label, sample_output]))

## Tune nhẹ và EXPORT CONFIG
Ga và steering thẳng `0` chỉ hiển thị và bị khóa. Có thể sửa 3 phase LEFT/RIGHT và từng thời gian cạnh trước khi lưu. Đây là config runner sẽ đọc.

In [ ]:
center_w = widgets.FloatText(value=0.0, description='Straight=0', disabled=True)
left_pre_w = widgets.BoundedFloatText(value=float(config['turns']['left_90'].get('pre_steer_time',0.15)), min=0.0, max=2.0, step=0.01, description='LEFT pre')
left_steer_w = widgets.BoundedFloatText(value=float(config['turns']['left_90']['steering']), min=-1.0, max=0.0, step=0.01, description='LEFT steer')
left_time_w = widgets.BoundedFloatText(value=float(config['turns']['left_90']['time']), min=0.0, max=5.0, step=0.01, description='LEFT time')
left_center_w = widgets.BoundedFloatText(value=float(config['turns']['left_90'].get('center_settle_time',0.15)), min=0.0, max=2.0, step=0.01, description='LEFT center')
right_pre_w = widgets.BoundedFloatText(value=float(config['turns']['right_90'].get('pre_steer_time',0.15)), min=0.0, max=2.0, step=0.01, description='RIGHT pre')
right_steer_w = widgets.BoundedFloatText(value=float(config['turns']['right_90']['steering']), min=0.0, max=1.0, step=0.01, description='RIGHT steer')
right_time_w = widgets.BoundedFloatText(value=float(config['turns']['right_90']['time']), min=0.0, max=5.0, step=0.01, description='RIGHT time')
right_center_w = widgets.BoundedFloatText(value=float(config['turns']['right_90'].get('center_settle_time',0.15)), min=0.0, max=2.0, step=0.01, description='RIGHT center')
edge_widgets = {}
for a, b in EDGES:
    key = edge_key(a,b)
    edge_widgets[key] = widgets.BoundedFloatText(value=float(config['edges'].get(key,0)), min=0.0, max=20.0, step=0.01, description=key)
export_button = widgets.Button(description='EXPORT CONFIG', button_style='primary', icon='save')
export_status = widgets.HTML()

def export_config(_):
    driver.stop(center=True); arm.value = False
    config['motion']['throttle'] = FIXED_THROTTLE
    config['motion']['steering_center'] = 0.0
    config['turns']['left_90'] = {'pre_steer_time': float(left_pre_w.value), 'steering': float(left_steer_w.value), 'time': float(left_time_w.value), 'center_settle_time': float(left_center_w.value)}
    config['turns']['right_90'] = {'pre_steer_time': float(right_pre_w.value), 'steering': float(right_steer_w.value), 'time': float(right_time_w.value), 'center_settle_time': float(right_center_w.value)}
    for key, widget in edge_widgets.items(): config['edges'][key] = float(widget.value)
    try:
        save_config(config, CONFIG_PATH, require_measurements=True)
        export_status.value = f'<b style="color:#080">Saved: {CONFIG_PATH}</b>'
    except Exception as exc:
        export_status.value = f'<b style="color:#b00">Not saved: {exc}</b>'

export_button.on_click(export_config)
edge_box = widgets.GridBox(list(edge_widgets.values()), layout=widgets.Layout(grid_template_columns='repeat(3, 170px)', grid_gap='6px'))
display(widgets.VBox([widgets.HTML(f'<b>Throttle {FIXED_THROTTLE:.2f} — LOCKED</b>'), center_w, widgets.HTML('<b>3 phases: PRE-STEER → DRIVE ARC → CENTER</b>'), widgets.HBox([left_pre_w,left_steer_w,left_time_w,left_center_w]), widgets.HBox([right_pre_w,right_steer_w,right_time_w,right_center_w]), widgets.HTML('<b>Edge times</b>'), edge_box, export_button, export_status]))

## Dừng an toàn — luôn chạy cell này trước khi đóng notebook

In [ ]:
driver.stop(center=True)
arm.value = False
print('Stopped safely; throttle=0, steering=center, DISARMED.')